# Figures of the v3 manuscript

Single-column figures (8.7 cm wide, 400 dpi) drawn only from `paper/data/` and `paper/data/v3/` (run `analysis_v3.ipynb` first). Written to `paper/figures/v3/`. Fig. 1 is v1's memory paradigm (`trace_fit.pdf`). Cohort colours are v1's.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path.cwd().resolve().parent                       # this notebook lives in paper/
DATA, D3, OUT = REPO / "paper/data", REPO / "paper/data/v3", REPO / "paper/figures/v3"
OUT.mkdir(parents=True, exist_ok=True)
W1 = 3.42                                                # PNAS single column, 8.7 cm
INK, INK2 = "#1a1a1a", "#666666"
ORDER = ["dHCP", "BCP", "CALM", "RED", "ACE", "HCPd", "HCPya", "camCAN", "HCPa"]
COH = dict(zip(ORDER, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#7f7f7f", "#bcbd22", "#17becf"]))  # cohort colours of v1

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 6.5, "axes.labelsize": 6.5, "axes.titlesize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 5.8, "legend.title_fontsize": 5.8, "axes.linewidth": 0.5, "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK, "text.color": INK, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.minor.width": 0.4, "ytick.minor.width": 0.4, "xtick.major.size": 2.2, "ytick.major.size": 2.2,
    "xtick.minor.size": 1.2, "ytick.minor.size": 1.2, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5, "axes.labelpad": 1.5,
    "lines.linewidth": 0.9, "legend.frameon": False, "legend.handlelength": 1.4, "legend.handletextpad": 0.4,
    "legend.borderaxespad": 0.2, "legend.labelspacing": 0.25, "pdf.fonttype": 42, "savefig.dpi": 400, "mathtext.fontset": "stix"})


def letter(ax, s, x=-0.34, y=1.0):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom", ha="left")


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", pad_inches=0.02, dpi=400)


def r_(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def age_means(d, col):
    """Mean of col per cohort and age: 1-year bins, 3-month bins below 2 years (as in v1)."""
    a = np.where(d.age < 2, np.floor(d.age * 4) / 4 + 0.125, np.floor(d.age) + 0.5)
    return d.assign(abin=a).groupby(["cohort", "abin"])[col].mean().reset_index()


def trajectories(ax, d, col, ylabel):
    m = age_means(d, col)
    for c in ORDER:
        g = m[m.cohort == c]
        ax.plot(g.abin, g[col], "-o", color=COH[c], ms=1.5, lw=0.45, mew=0)
    ax.set(xlabel="age (years)", ylabel=ylabel, xlim=(-2, 92), xticks=[0, 20, 40, 60, 80])

In [ ]:
d = pd.read_csv(D3 / "subjects_v3.csv")                      # one row per connectome (analysis_v3.ipynb)
v2 = pd.read_csv(DATA / "subject_v2.csv"); ex = pd.read_csv(DATA / "examples.csv")
nl = pd.read_csv(D3 / "nulls_v3.csv"); thr = pd.read_csv(D3 / "threshold.csv"); kap = pd.read_csv(D3 / "kappa.csv")
uk = pd.read_csv(D3 / "upsilon_k.csv"); nodes = pd.read_csv(D3 / "nodes_sample.csv")
N = json.load(open(D3 / "numbers_v3.json"))

## Fig. 2: memory counts input directions, the stable rank sets the count
(a) closed form vs simulation; (b) Gramian spectra and the noise floor; (c) MC vs stable rank; (d) factor shares of the stable rank; (e) null models; (f) thresholding vs binarisation.

In [ ]:
fig = plt.figure(figsize=(W1, 4.75))
gs = fig.add_gridspec(3, 2, left=0.13, right=0.985, top=0.965, bottom=0.085, hspace=0.52, wspace=0.62)

# (a) closed form against the simulated reservoir
ax = fig.add_subplot(gs[0, 0])
x, y = v2["MClin_s1e-05"], v2["MC_s1e-05"]
cm = mpl.colors.LinearSegmentedColormap.from_list("b", ["#c6dbef", "#4292c6", "#08306b"])
ax.hexbin(x, y, gridsize=30, cmap=cm, bins="log", mincnt=1, linewidths=0.15, edgecolors="face", rasterized=True)
lo, hi = np.floor(min(x.min(), y.min())), np.ceil(max(x.max(), y.max()))
ax.plot([lo, hi], [lo, hi], color=INK2, lw=0.5, ls="--")
ax.text(0.05, 0.95, f"$r$ = {r_(x, y):.3f}\n$n$ = {len(x):,}", transform=ax.transAxes, va="top")
ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel="MC, closed form", ylabel="MC, simulated reservoir"); letter(ax, "a")

# (b) variance of the input-driven directions against the noise floor
ax = fig.add_subplot(gs[0, 1])
ax.axhspan(1e-30, 1, color="#efefef", lw=0, zorder=0)
mc = v2.set_index("sid")["MC_s1e-05"]
for lab, col, name in (("p5", "#9ecae1", "5th"), ("p50", "#4292c6", "50th"), ("p95", "#08306b", "95th")):
    e = ex[(ex.label == lab) & (ex.k <= 24)]
    ax.semilogy(e.k, e.c_over_eta, "-o", color=col, ms=1.6, lw=0.8, mew=0, label=f"MC {mc.loc[e.sid.iloc[0]]:.1f}")
ax.axhline(1, color=INK, lw=0.6, ls="--")
ax.text(23.5, 3e-9, "below the noise floor", ha="right", va="bottom", fontsize=5.5, color=INK2)
ax.set(xlabel="input-driven direction $k$", ylabel="variance / noise floor, $c_k/\\eta$", xlim=(0, 24.5), ylim=(1e-9, 1e12),
       xticks=[1, 5, 10, 15, 20])
ax.yaxis.set_major_locator(mpl.ticker.LogLocator(numticks=6)); ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.legend(loc="upper right"); letter(ax, "b")

# (c) memory against the stable rank, coloured by cohort
ax = fig.add_subplot(gs[1, 0])
for c in ORDER:
    g = d[d.cohort == c]
    ax.scatter(g.rs, g.MC, s=0.7, color=COH[c], lw=0, alpha=0.55, rasterized=True)
b = np.polyfit(np.log(d.rs), d.MC, 1); xs = np.geomspace(d.rs.min(), d.rs.max(), 50)
ax.plot(xs, np.polyval(b, np.log(xs)), color=INK, lw=0.8)
ax.set_xscale("log"); ax.set_xticks([3, 4, 6, 10]); ax.xaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.text(0.05, 0.95, f"$r$ = {r_(np.log(d.rs), d.MC):.2f}", transform=ax.transAxes, va="top")
ax.set(xlabel="stable rank, $r_s=\\|W\\|_F^2/\\rho^2$", ylabel="memory capacity"); letter(ax, "c")

# (d) what builds the stable rank across connectomes: shares of the exact factors
ax = fig.add_subplot(gs[1, 1]); fs = N["factor_shares"]; rows = ["all"] + ORDER
for f, mk, col, lab in (("G", "o", INK, "$G$"), ("D", "s", "#a3a3a3", "$D$"), ("L", "^", "#e6550d", "$L$")):
    ax.scatter([fs[r][f] for r in rows], np.arange(len(rows)), marker=mk, s=8, color=col, lw=0, label=lab, zorder=3)
ax.axvline(0, color=INK2, lw=0.5); ax.axvline(1, color=INK2, lw=0.5, ls=":")
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=5.6); ax.invert_yaxis(); ax.tick_params(axis="y", length=0)
for t, r in zip(ax.get_yticklabels(), rows):
    t.set_color(COH.get(r, INK)); t.set_fontweight("bold" if r == "all" else "normal")
ax.set(xlabel="share of the variance of log $r_s$", xlim=(-0.65, 1.8))
ax.legend(loc="lower center", bbox_to_anchor=(0.45, 0.99), ncol=3, columnspacing=1.2, handletextpad=0.05); letter(ax, "d", x=-0.42)

# (e) null models: observed change in memory and the change predicted by the stable rank
ax = fig.add_subplot(gs[2, 0])
names = {"Uniform": "Uniform", "BrokenStick": "Broken stick", "Reshuffle": "Reshuffle", "EqualStrength": "Equal strength",
         "InterHalf": "Interhem. ×½"}
NC = dict(zip(names.values(), ["#e6550d", "#fdae6b", "#969696", "#3182bd", "#756bb1"]))
q = nl[nl.model.isin(list(names))].assign(null=lambda t: t.model.map(names)); lab = list(names.values())
ax.axvline(0, color=INK2, lw=0.5)
sns.boxplot(data=q, y="null", x="dMC", order=lab, color="white", width=0.6, fliersize=0, linewidth=0.5, ax=ax,
            boxprops=dict(edgecolor=INK2), whiskerprops=dict(color=INK2), medianprops=dict(color=INK), capprops=dict(color=INK2))
sns.stripplot(data=q, y="null", x="dMC", order=lab, hue="null", palette=NC, size=1.4, jitter=0.2, alpha=0.8, ax=ax,
              legend=False, rasterized=True)
pred = q.groupby("null").dMC_pred.median().reindex(lab)
ax.scatter(pred.values, np.arange(len(lab)), marker="D", s=9, color=INK, edgecolor="white", lw=0.4, zorder=5)
ax.annotate("predicted\nfrom $r_s$", xy=(pred["Equal strength"], 3), xytext=(pred["Equal strength"], 1.6), ha="center", va="center",
            fontsize=5.4, arrowprops=dict(arrowstyle="-", lw=0.4, color=INK2, shrinkB=2))
ax.set(xlabel="$\\Delta$MC (null $-$ real)", ylabel=""); ax.tick_params(axis="y", length=0)
for t in ax.get_yticklabels(): t.set_fontsize(5.6)
letter(ax, "e", x=-0.5)

# (f) keep the weights: thresholding keeps memory, binarising erases it; the disparity ratio needs the weak links
ax = fig.add_subplot(gs[2, 1])
ref = thr[(thr.kind == "weighted") & (thr.frac == 1)].set_index("sid")
for kind, col_, ref_col, col, ls, mk in (("weighted", "MC", "MC", "#3182bd", "-", "o"), ("binary", "MC", "MC", "#e6550d", "-", "o"),
                                        ("weighted", "ratio", "ratio", "#737373", "--", "s")):
    rr = {f: r_(g.set_index("sid")[col_].loc[ref.index], ref[ref_col]) for f, g in thr[thr.kind == kind].groupby("frac")}
    ax.plot(list(rr), list(rr.values()), ls=ls, marker=mk, color=col, ms=2.3, mew=0, lw=1)
ax.axhline(0, color=INK2, lw=0.5)
ax.set_xscale("log"); ax.set_xticks([1, 0.5, 0.25, 0.1]); ax.set_xticklabels(["1", "0.5", "0.25", "0.1"]); ax.minorticks_off()
ax.set_xlim(1.25, 0.08)
ax.text(1.2, 1.06, "MC, weights kept", color="#3182bd", fontsize=5.8, va="bottom")
ax.text(1.2, -0.25, "MC, binarised", color="#e6550d", fontsize=5.8, va="top")
ax.text(1.2, 0.6, "$\\langle\\Upsilon/\\Upsilon_{null}\\rangle$,\nweights kept", color="#737373", fontsize=5.8, ha="left", va="center")
ax.set(xlabel="fraction of connections kept\n(strongest first)", ylabel="$r$ with the full\nweighted connectome", ylim=(-0.55, 1.22))
letter(ax, "f", x=-0.42)
for a in fig.axes:
    sns.despine(ax=a)
save(fig, "fig2_memory")

## Fig. 3: lifespan
(a) nodal disparity vs degree (v1); (b–d) lifespan trajectories; (e) MC vs homotopic share; (f) homotopic share across the lifespan; (g) within-cohort aging; (h) virtual aging of young connectomes.

In [ ]:
fig = plt.figure(figsize=(W1, 6.2))
gs = fig.add_gridspec(4, 2, left=0.13, right=0.985, top=0.985, bottom=0.085, hspace=0.5, wspace=0.6)

# (a) nodal disparity against degree (v1 Fig. 4a)
ax = fig.add_subplot(gs[0, 0])
ax.scatter(nodes.k, nodes.Ups, s=0.5, color="#6baed6", alpha=0.18, lw=0, rasterized=True)
k = np.arange(1, 90.0); mu = 2 * k / (k + 1)
sd = k * np.sqrt((20 + 4 * k) / ((k + 1) * (k + 2) * (k + 3)) - 4 / (k + 1) ** 2)
ax.fill_between(k, np.clip(mu - 2 * sd, 0.7, None), mu + 2 * sd, color="#bdbdbd", alpha=0.8, lw=0, label="null $\\pm2\\sigma$")
ax.plot(k, mu, color="#636363", lw=0.7)
ax.plot(k, k, color="#d62728", lw=0.7, ls="--", label="$\\Upsilon=k$")
g = uk[uk["size"] >= 30]
ax.plot(g.k, g["mean"], color="#08519c", lw=1.2, label=f"$\\langle\\Upsilon\\rangle_k\\propto k^{{{N['upsilon_alpha']:.2f}}}$")
ax.set(xscale="log", yscale="log", xlabel="degree $k$", ylabel="nodal disparity, $\\Upsilon=kY$", xlim=(1, 90), ylim=(0.7, 90))
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_locator(mpl.ticker.FixedLocator([1, 3, 10, 30, 90])); axis.set_major_formatter(mpl.ticker.ScalarFormatter())
    axis.set_minor_locator(mpl.ticker.NullLocator())
ax.legend(loc="upper left"); letter(ax, "a")

# (b-d, f) lifespan trajectories, per-age means of each cohort
for pos, col, lab, let in (((0, 1), "Ratio", "disparity, $\\langle\\Upsilon/\\Upsilon_{null}\\rangle$", "b"),
                           ((1, 0), "MC", "memory capacity", "c"),
                           ((1, 1), "rs", "stable rank $r_s$", "d"),
                           ((2, 1), "hom_F2", "homotopic share $h$", "f")):
    ax = fig.add_subplot(gs[pos]); trajectories(ax, d, col, lab); letter(ax, let)

# (e) memory against the homotopic share, coloured by age
ax = fig.add_subplot(gs[2, 0])
sc = ax.scatter(d.hom_F2, d.MC, c=d.age, cmap="viridis", vmin=0, vmax=90, s=0.7, lw=0, alpha=0.65, rasterized=True)
cax = ax.inset_axes([0.56, 0.9, 0.4, 0.045]); cb = fig.colorbar(sc, cax=cax, orientation="horizontal")
cb.set_ticks([0, 45, 90]); cb.ax.tick_params(labelsize=5, length=1.5, pad=1, width=0.4); cb.outline.set_linewidth(0.3)
cax.set_title("age (years)", fontsize=5.5, pad=1.5)
ax.text(0.05, 0.05, f"$r$ = {r_(d.hom_F2, d.MC):.2f}", transform=ax.transAxes)
ax.set(xlabel="homotopic share $h$", ylabel="memory capacity", ylim=(d.MC.min() - 0.2, d.MC.max() + 0.9)); letter(ax, "e")

# (g) within camCAN and HCPa: the homotopic share accounts for the aging rise of memory
ax = fig.add_subplot(gs[3, 0]); L = N["lifespan"]; w = 0.27
bars = (("r_age_hom", "$h$", "#6a3d9a"), ("r_age_MC", "MC", INK), ("pr_age_MC_given_hom", "MC | $h$", "#bdbdbd"))
for j, (key, lab, col) in enumerate(bars):
    ax.bar(np.arange(2) + (j - 1) * w, [L[c][key] for c in ("camCAN", "HCPa")], width=w * 0.92, color=col, edgecolor=INK, lw=0.3, label=lab)
ax.axhline(0, color=INK, lw=0.5); ax.set_xticks([0, 1]); ax.set_xticklabels(["camCAN", "HCPa"]); ax.tick_params(axis="x", length=0)
for t, c in zip(ax.get_xticklabels(), ("camCAN", "HCPa")):
    t.set_color(COH[c]); t.set_fontweight("bold")
ax.set(ylabel="correlation with age", ylim=(-0.72, 0.6), xlim=(-0.5, 1.5))
ax.legend(loc="upper center", ncol=3, handlelength=0.9, columnspacing=0.8); letter(ax, "g")

# (h) virtual aging: young connectomes with rescaled interhemispheric weights
ax = fig.add_subplot(gs[3, 1]); VR = N["virtual_rescaling"]
for c in ("camCAN", "HCPa"):
    g = kap[kap.cohort == c].groupby("kappa").MC.agg(["mean", "sem"])
    ax.fill_between(g.index, g["mean"] - 1.96 * g["sem"], g["mean"] + 1.96 * g["sem"], color=COH[c], alpha=0.25, lw=0)
    ax.plot(g.index, g["mean"], "-o", color=COH[c], ms=1.8, mew=0, lw=0.9)
    ax.axhline(VR[c]["MC_old"], color=COH[c], lw=0.7, ls="--")
    Fy, Fo = VR[c]["inter_young"], VR[c]["inter_old"]                # interhemispheric fraction of the total weight
    ax.axvline(Fo * (1 - Fy) / (Fy * (1 - Fo)), color=COH[c], lw=0.6, ls=":")   # kappa that brings Fy to Fo
ax.text(1.5, VR["camCAN"]["MC_old"] + 0.03, "observed, ≥ 70 y", ha="right", va="bottom", fontsize=5.4, color=INK2)
ax.text(0.47, 9.82, "$\\kappa$ matching\n≥ 70 y", ha="left", va="bottom", fontsize=5.4, color=INK2)
ax.set(xlabel="interhemispheric weights × $\\kappa$", ylabel="memory capacity"); letter(ax, "h")

hs = [mpl.lines.Line2D([], [], color=COH[c], marker="o", ms=2.5, lw=0.6, label=c) for c in ORDER]
fig.legend(handles=hs, loc="lower center", bbox_to_anchor=(0.56, 0.0), ncol=5, columnspacing=0.9, handletextpad=0.3, fontsize=6)
for a in fig.axes:
    if a is not cax:
        sns.despine(ax=a)
save(fig, "fig3_lifespan")